In [2]:
# ==========================================
# 01. Imports & Environment
# ==========================================

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torchvision

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# Select GPU if CUDA is available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Display environment information
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130
Torchvision: 0.29.1+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
# ==========================================
# 02. Load Dataset Splits
# ==========================================

# Locate the project root
PROJECT_ROOT = Path.cwd().parents[1]

# Directory containing our prepared datasets
DATA_DIR = PROJECT_ROOT / "03_Assets" / "Processed"

# Load training, validation, and test metadata
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# Verify that all datasets loaded correctly
print("Project root:", PROJECT_ROOT)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

# Check the first few entries
train_df.head()


Project root: c:\Users\jtmor\Documents\Project Hub\01_Active\Engineering\chest-xray-pneumonia-detection
Training images: 20778
Validation images: 4453
Test images: 4453


,SOPInstanceUID,PatientID,label,target,path,ViewPosition
0,1.2.276.0.7230010.3.1.4.8323329.4108.151787430...,a73a2519-e185-4eb7-95fe-03c39509e47d,No Lung Opacity / Not Normal,0,C:\XrayData\mdai_rsna_project_x9N20BZa_images_...,PA
1,1.2.276.0.7230010.3.1.4.8323329.19694.15178744...,0e530c69-319b-413d-a9f3-2eb0810bba05,Lung Opacity,1,C:\XrayData\mdai_rsna_project_x9N20BZa_images_...,PA
2,1.2.276.0.7230010.3.1.4.8323329.3265.151787430...,cb60942f-4ed7-4f81-90c0-cc1db42ccaeb,No Lung Opacity / Not Normal,0,C:\XrayData\mdai_rsna_project_x9N20BZa_images_...,PA
3,1.2.276.0.7230010.3.1.4.8323329.23777.15178744...,3da1b7e0-888d-4b1b-8304-10d7d609d9ed,Lung Opacity,1,C:\XrayData\mdai_rsna_project_x9N20BZa_images_...,AP
4,1.2.276.0.7230010.3.1.4.8323329.10499.15178743...,71639ce3-3715-4e4d-8bef-3df8bb5c6770,Normal,0,C:\XrayData\mdai_rsna_project_x9N20BZa_images_...,PA


In [4]:
# ==========================================
# 03. DICOM Image Loader
# ==========================================

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

        # Preprocessing for pretrained ResNet18
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        # Read DICOM image
        dicom = pydicom.dcmread(row["path"])
        image = dicom.pixel_array.astype(np.float32)

        # Correct inverted grayscale images
        if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() - image

        # Normalize pixel values to 0-255
        image -= image.min()
        image /= (image.max() + 1e-8)
        image = (image * 255).astype(np.uint8)

        # Convert grayscale image to RGB
        image = Image.fromarray(image).convert("RGB")

        # Apply preprocessing
        image = self.transform(image)

        # Binary classification label
        label = torch.tensor(
            int(row["target"]),
            dtype=torch.long
        )

        return image, label


# Create datasets
train_dataset = ChestXrayDataset(train_df)
val_dataset = ChestXrayDataset(val_df)
test_dataset = ChestXrayDataset(test_df)

# Test loading one image
sample_image, sample_label = train_dataset[0]

print("Image shape:", sample_image.shape)
print("Label:", sample_label.item())
print("Pixel range:", sample_image.min().item(), sample_image.max().item())


Image shape: torch.Size([3, 224, 224])
Label: 0
Pixel range: -2.1179039478302 2.4134206771850586


In [5]:
# ==========================================
# 04. Configure DataLoaders
# ==========================================

BATCH_SIZE = 32

# Start with zero workers for reliable execution
# in Jupyter notebooks on Windows.
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


Training batches: 650
Validation batches: 140
Test batches: 140


In [6]:
# ==========================================
# 05. Initialize ResNet18
# ==========================================

# Load pretrained ResNet18 weights
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Freeze pretrained feature extraction layers
for parameter in model.parameters():
    parameter.requires_grad = False

# Replace final layer for binary classification
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

# Move model to GPU
model = model.to(device)

# Define loss function and optimizer
# Weight classes to account for the imbalanced dataset
class_counts = train_df["target"].value_counts().sort_index()
class_weights = (
    len(train_df) / (2 * class_counts)
).to_numpy(dtype=np.float32)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, device=device)
)

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print("Model: ResNet18")
print("Output classes:", model.fc.out_features)
print("Class weights:", class_weights)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)


Model: ResNet18
Output classes: 2
Class weights: [0.6573652 2.088661 ]
Trainable parameters: 1026


In [7]:
# ==========================================
# 06. Train ResNet18
# ==========================================

import time

EPOCHS = 2

# Folder for trained model checkpoints
MODEL_DIR = PROJECT_ROOT / "04_Exports" / "Models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "resnet18_lung_opacity.pth"

best_val_loss = float("inf")
history = []

for epoch in range(EPOCHS):
    start_time = time.time()

    # Keep frozen backbone and BatchNorm statistics fixed.
    # Gradients still update the final classification layer.
    model.eval()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    # Training
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Batch {batch_idx + 1}/{len(train_loader)}",
                flush=True
            )

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    # Save best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch + 1,
            "val_loss": val_loss,
            "class_names": ["No Lung Opacity", "Lung Opacity"]
        }, MODEL_PATH)

        print("Best model saved.")

    elapsed = (time.time() - start_time) / 60

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS} completed "
        f"in {elapsed:.1f} minutes"
    )
    print(f"Train loss: {train_loss:.4f}")
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"Validation loss: {val_loss:.4f}")
    print(f"Validation accuracy: {val_acc:.4f}")
    print("-" * 40)

print("\nTraining finished.")
print("Model saved at:", MODEL_PATH)


Epoch 1/2 | Batch 100/650
Epoch 1/2 | Batch 200/650
Epoch 1/2 | Batch 300/650
Epoch 1/2 | Batch 400/650
Epoch 1/2 | Batch 500/650
Epoch 1/2 | Batch 600/650
Best model saved.

Epoch 1/2 completed in 14.1 minutes
Train loss: 0.5384
Train accuracy: 0.7376
Validation loss: 0.5564
Validation accuracy: 0.8066
----------------------------------------
Epoch 2/2 | Batch 100/650
Epoch 2/2 | Batch 200/650
Epoch 2/2 | Batch 300/650
Epoch 2/2 | Batch 400/650
Epoch 2/2 | Batch 500/650
Epoch 2/2 | Batch 600/650
Best model saved.

Epoch 2/2 completed in 13.4 minutes
Train loss: 0.5051
Train accuracy: 0.7593
Validation loss: 0.4917
Validation accuracy: 0.7487
----------------------------------------

Training finished.
Model saved at: c:\Users\jtmor\Documents\Project Hub\01_Active\Engineering\chest-xray-pneumonia-detection\04_Exports\Models\resnet18_lung_opacity.pth
